In [ ]:
#| default_exp tenant

In [ ]:
#| hide
from nbdev.showdoc import *

# tenant
> One call to host N apps on a single VPS — each on its own subdomain, routed live via SQLite.

In [ ]:
#| export
from fastcore.all import Path, L
from dockeasy.core import detect_app, Compose
from dockeasy.proxy import caddy_sqlite_dockerfile, caddy_sqlite_svc, cloudflared_svc
import sqlite3, re

## Test helpers

In [ ]:
import tempfile, shutil, os
from fastcore.test import test_eq

def _mk_tenant(tmp, name, framework='fasthtml'):
    'Create a minimal tenant app dir for tests'
    d = Path(tmp)/name
    d.mkdir()
    if framework == 'fasthtml':
        (d/'pyproject.toml').write_text('[project]\nname="app"\ndependencies=["python-fasthtml"]')
    elif framework == 'go':   (d/'go.mod').write_text('module app\ngo 1.22')
    elif framework == 'rust': (d/'Cargo.toml').write_text('[package]\nname="app"')
    elif framework == 'node': (d/'package.json').write_text('{}')
    return d

## Route detection

`_detect_port` runs `detect_app()` on a tenant directory and reads the `EXPOSE` instruction to find the port the app actually listens on — 5001 for FastHTML, 8080 for Go/Rust, 3000 for Node.

In [ ]:
#| export
def _detect_port(path):
    'Infer app port from project files by reading EXPOSE from the detected Dockerfile'
    m = re.search(r'^EXPOSE (\d+)', str(detect_app(path)), re.MULTILINE)
    return int(m.group(1)) if m else 5001

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    test_eq(_detect_port(_mk_tenant(tmp, 'fh',   'fasthtml')), 5001)
    test_eq(_detect_port(_mk_tenant(tmp, 'go',   'go')),       8080)
    test_eq(_detect_port(_mk_tenant(tmp, 'rust', 'rust')),     8080)
    test_eq(_detect_port(_mk_tenant(tmp, 'node', 'node')),     3000)
    print('_detect_port() OK')

## sync_routes

Scans `tenants_dir` for app subdirectories, creates the routes DB if needed, and upserts one row per tenant: `(domain, host, port)`. Safe to call on every deploy — idempotent. Use `prune=True` to remove rows for directories that no longer exist.

In [ ]:
#| export
def sync_routes(tenants_dir='./tenants', db='routes.db', prune=False):
    'Scan tenants_dir for app subdirs and upsert routes DB. prune=True removes rows for missing dirs.'
    tenants_dir = Path(tenants_dir)
    subdirs = {d.name: d for d in sorted(tenants_dir.iterdir()) if d.is_dir() and not d.name.startswith('.')}
    con = sqlite3.connect(db)
    con.execute('''CREATE TABLE IF NOT EXISTS routes
                   (domain TEXT PRIMARY KEY, host TEXT NOT NULL, port INTEGER NOT NULL)''')
    existing = {r[0] for r in con.execute('SELECT domain FROM routes')}
    for name, d in subdirs.items():
        if name not in existing:
            con.execute('INSERT INTO routes VALUES (?,?,?)', (name, name, _detect_port(d)))
    if prune:
        for name in existing - subdirs.keys():
            con.execute('DELETE FROM routes WHERE domain = ?', (name,))
    con.commit()
    result = {r[0]: (r[1], r[2]) for r in con.execute('SELECT domain, host, port FROM routes')}
    con.close()
    return result

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme',   'fasthtml')
    _mk_tenant(tdir, 'globex', 'fasthtml')
    db = f'{tmp}/routes.db'
    routes = sync_routes(tdir, db)
    test_eq(set(routes), {'acme', 'globex'})
    test_eq(routes['acme'],   ('acme',   5001))
    test_eq(routes['globex'], ('globex', 5001))
    print('sync_routes() basic OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    db = f'{tmp}/routes.db'
    sync_routes(tdir, db)
    sync_routes(tdir, db)  # re-run — must not duplicate
    con = sqlite3.connect(db)
    test_eq(con.execute('SELECT COUNT(*) FROM routes').fetchone()[0], 1)
    con.close()
    print('sync_routes() idempotent OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'webapp', 'fasthtml')
    _mk_tenant(tdir, 'api',    'go')
    routes = sync_routes(tdir, f'{tmp}/routes.db')
    test_eq(routes['webapp'][1], 5001)
    test_eq(routes['api'][1],    8080)
    print('sync_routes() mixed frameworks OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    db = f'{tmp}/routes.db'
    sync_routes(tdir, db)
    shutil.rmtree(tdir/'acme')
    routes = sync_routes(tdir, db, prune=False)
    assert 'acme' in routes, 'prune=False must keep rows for removed dirs'
    print('sync_routes() prune=False OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme',   'fasthtml')
    _mk_tenant(tdir, 'globex', 'fasthtml')
    db = f'{tmp}/routes.db'
    sync_routes(tdir, db)
    shutil.rmtree(tdir/'acme')
    routes = sync_routes(tdir, db, prune=True)
    assert 'acme' not in routes
    assert 'globex' in routes
    print('sync_routes() prune=True OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    (tdir/'.git').mkdir()                          # hidden dir — ignored
    (tdir/'docker-compose.yml').write_text('')     # file (not dir) — ignored
    routes = sync_routes(tdir, f'{tmp}/routes.db')
    test_eq(set(routes), {'acme'})
    print('sync_routes() edge cases OK')

## tenant_stack

The top-level entry point. Calls `sync_routes`, writes missing Dockerfiles via `detect_app`,
creates the `.caddy-sqlite/` build context, and returns a fully wired `Compose` ready to
`.save()` and `.up()`.

The Caddy service has **both** `image:` and `build:` set — Docker Compose uses the pre-built
GHCR image when available (pulled in CI), and falls back to building locally via xcaddy otherwise.

In [ ]:
#| export
CADDY_SQLITE_IMAGE = 'ghcr.io/vedicreader/caddy-sqlite:latest'
_CADDY_BUILD_DIR   = '.caddy-sqlite'

def tenant_stack(base_domain, tenants_dir='./tenants', *,
                 db='routes.db', cloudflared=True, crowdsec=False,
                 conf='Caddyfile', caddy_image=CADDY_SQLITE_IMAGE,
                 prune=False, **kw):
    '''Multi-tenant compose stack — scans tenants_dir, syncs routes DB, builds missing Dockerfiles,
    and returns a wired Compose with caddy-sqlite + optional cloudflared/crowdsec.

    caddy_image defaults to ghcr.io/vedicreader/caddy-sqlite:latest (pre-built in CI via
    .github/workflows/caddy-sqlite.yml). If the image is not cached locally, compose builds
    it from .caddy-sqlite/Dockerfile using xcaddy (takes 3-5 min on first run).'''
    tenants_dir = Path(tenants_dir)
    routes = sync_routes(tenants_dir, db, prune=prune)
    for name in routes:
        d = tenants_dir/name
        if not (d/'Dockerfile').exists():
            detect_app(str(d)).save(d/'Dockerfile')
    caddy_dir = Path(_CADDY_BUILD_DIR)
    caddy_dir.mkdir(exist_ok=True)
    caddy_sqlite_dockerfile().save(caddy_dir/'Dockerfile')
    dc = Compose()
    for name in routes:
        dc = dc.svc(name, build=str(tenants_dir/name), networks=['web'], restart='unless-stopped')
    caddy_kw = caddy_sqlite_svc(f'*.{base_domain}', db='/routes.db',
                                 image=caddy_image, build=str(caddy_dir),
                                 crowdsec=crowdsec, conf=conf)
    caddy_kw['volumes'].append(f'{Path(db).resolve()}:/routes.db')
    dc = dc.svc('caddy', **caddy_kw)
    if cloudflared: dc = dc.svc('cloudflared', **cloudflared_svc())
    return dc.network('web').volume('caddy_data').volume('caddy_config')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme',   'fasthtml')
    _mk_tenant(tdir, 'globex', 'fasthtml')
    dc = tenant_stack('example.com', tdir, db=f'{tmp}/routes.db',
                      conf=f'{tmp}/Caddyfile', cloudflared=False)
    d = dc.to_dict()
    assert 'acme'        in d['services']
    assert 'globex'      in d['services']
    assert 'caddy'       in d['services']
    assert 'cloudflared' not in d['services']
    print('tenant_stack() structure OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    conf = f'{tmp}/Caddyfile'
    dc = tenant_stack('example.com', tdir, db=f'{tmp}/routes.db',
                      conf=conf, cloudflared=False)
    caddy = dc.to_dict()['services']['caddy']
    test_eq(caddy['image'], CADDY_SQLITE_IMAGE)
    assert _CADDY_BUILD_DIR in caddy.get('build', ''), f'build: missing .caddy-sqlite, got: {caddy}'
    cf = Path(conf).read_text()
    assert '*.example.com' in cf
    assert 'sqlite_router'  in cf
    print('tenant_stack() caddy service + Caddyfile OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    db = f'{tmp}/routes.db'
    dc = tenant_stack('example.com', tdir, db=db,
                      conf=f'{tmp}/Caddyfile', cloudflared=False)
    vols = dc.to_dict()['services']['caddy']['volumes']
    db_abs = str(Path(db).resolve())
    assert any('/routes.db' in v and db_abs in v for v in vols), \
        f'routes DB bind-mount not found in: {vols}'
    print('tenant_stack() DB mount OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    # cloudflared=True adds service
    dc = tenant_stack('example.com', tdir, db=f'{tmp}/routes1.db',
                      conf=f'{tmp}/Caddyfile1', cloudflared=True)
    assert 'cloudflared' in dc.to_dict()['services']
    # crowdsec=True adds block in Caddyfile
    dc2 = tenant_stack('example.com', tdir, db=f'{tmp}/routes2.db',
                       conf=f'{tmp}/Caddyfile2', cloudflared=False, crowdsec=True)
    assert 'crowdsec' in Path(f'{tmp}/Caddyfile2').read_text()
    print('tenant_stack() cloudflared + crowdsec flags OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    globex = _mk_tenant(tdir, 'globex', 'fasthtml')
    sentinel = 'FROM scratch\n'
    (globex/'Dockerfile').write_text(sentinel)           # pre-existing
    tenant_stack('example.com', tdir, db=f'{tmp}/routes.db',
                 conf=f'{tmp}/Caddyfile', cloudflared=False)
    assert (tdir/'acme'/'Dockerfile').exists()            # generated
    assert (tdir/'globex'/'Dockerfile').read_text() == sentinel  # NOT overwritten
    print('tenant_stack() Dockerfiles OK')

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    _mk_tenant(tdir, 'acme', 'fasthtml')
    orig = os.getcwd()
    try:
        os.chdir(tmp)
        tenant_stack('example.com', tdir, db=f'{tmp}/routes.db',
                     conf=f'{tmp}/Caddyfile', cloudflared=False)
        df_path = Path(tmp)/'.caddy-sqlite'/'Dockerfile'
        assert df_path.exists()
        assert 'xcaddy' in df_path.read_text()
    finally:
        os.chdir(orig)
    print('tenant_stack() .caddy-sqlite/ OK')

## Incremental deployment

The SQLite router's key advantage: **adding or removing a tenant doesn't restart Caddy**. `caddy-sqlite-router` queries the DB on every request — a new `INSERT` is live immediately, a `DELETE` stops routing at once.

| Action | Steps | Downtime |
|--------|-------|---------|
| Initial stack | `tenant_stack()` → `dc.save()` → `dc.up()` | none (first deploy) |
| Add tenant | add dir → `sync_routes()` → `docker compose up -d --build <name>` | zero |
| Remove tenant | `sync_routes(prune=True)` → `docker compose stop <name>` | zero |

The `deploy-tenant.yml` workflow at `.github/workflows/deploy-tenant.yml` automates the "add tenant" path on every push to `tenants/**`.

In [ ]:
# Three-phase incremental deployment simulation (no Docker required)
with tempfile.TemporaryDirectory() as tmp:
    tdir = Path(tmp)/'tenants'; tdir.mkdir()
    db = f'{tmp}/routes.db'

    # Phase 1: initial stack — 2 tenants
    _mk_tenant(tdir, 'acme',   'fasthtml')
    _mk_tenant(tdir, 'globex', 'fasthtml')
    routes = sync_routes(tdir, db)
    test_eq(set(routes), {'acme', 'globex'})
    print('Phase 1: 2 tenants live')

    # Phase 2: new tenant — drop a directory, call sync_routes()
    # Caddy reads the DB on every request: new row → new subdomain live immediately
    _mk_tenant(tdir, 'initech', 'go')
    routes = sync_routes(tdir, db)          # INSERT only; existing rows untouched
    test_eq(set(routes), {'acme', 'globex', 'initech'})
    test_eq(routes['initech'], ('initech', 8080))
    print('Phase 2: initech (Go) added — Caddy live immediately, no restart')

    # Phase 3: tenant leaves — prune removes the row, routing stops on next request
    shutil.rmtree(tdir/'globex')
    routes = sync_routes(tdir, db, prune=True)
    test_eq(set(routes), {'acme', 'initech'})
    print('Phase 3: globex pruned — routing stopped immediately')

print('incremental deployment simulation OK')

In [ ]:
#| eval: false
# On your VPS after the initial stack is running:

# ── add a new tenant (zero downtime) ─────────────────────────────────────────
# 1. Deploy tenant code (git clone / rsync / etc.)
# Path('./tenants/initech').mkdir()
# subprocess.run(['git', 'clone', 'git@github.com:...', './tenants/initech'])

# 2. Sync routes DB — Caddy picks up the new row on the very next request
import subprocess
routes = sync_routes('./tenants', 'routes.db')
print(f'Active tenants: {list(routes)}')   # ['acme', 'globex', 'initech']

# 3. Start only the new container — other services keep running uninterrupted
subprocess.run(['docker', 'compose', 'up', '-d', '--build', 'initech'], check=True)
print('initech.yourdomain.com is live')

# ── remove a tenant (zero downtime) ──────────────────────────────────────────
# shutil.rmtree('./tenants/globex')
# sync_routes('./tenants', 'routes.db', prune=True)   # row deleted → routing stops immediately
# subprocess.run(['docker', 'compose', 'stop', 'globex'], check=True)

## GitHub Actions: caddy-sqlite image

The workflow at `.github/workflows/caddy-sqlite.yml` builds and pushes `ghcr.io/vedicreader/caddy-sqlite:latest`
to GHCR whenever `proxy.py` or `01_proxy.ipynb` changes on `main`. This means the slow xcaddy build
runs once in CI — VPS deploys just pull the pre-built image.

In [ ]:
#| eval: false
# Show the workflow file — not executed by nbdev-test
workflow = open('.github/workflows/caddy-sqlite.yml').read()
print(workflow)

## Example: 3-tenant stack on a single VPS

Live test — creates three FastHTML apps, syncs routes, and verifies each subdomain routes to the correct app.

**Prerequisites:**
- `BASE_DOMAIN` env var (e.g. `angalama.com`)
- `CF_TUNNEL_TOKEN` env var
- Wildcard tunnel ingress rule: `*.BASE_DOMAIN → http://caddy`
- Wildcard DNS CNAME: `*.BASE_DOMAIN → <tunnel-id>.cfargotunnel.com`

In [ ]:
#| eval: false
import os, time
from fastcore.all import urlread

BASE_DOMAIN = 'angalama.com'

# ── tenant apps ───────────────────────────────────────────────────────────────
work_dir = Path(tempfile.mkdtemp(dir='.'))
tenants  = work_dir/'tenants'
tenants.mkdir()

for name in ('acme', 'globex', 'initech'):
    d = tenants/name; d.mkdir()
    (d/'app.py').write_text(f'''
from fasthtml.common import *
app, rt = fast_app()
@rt('/')
def get(): return Titled('{name}', P('routed to {name} via SQLite ✓'))
serve()
''')
    (d/'pyproject.toml').write_text('''
[project]
name = "app"
version = "0.1.0"
dependencies = ["python-fasthtml", "starlette<0.46"]
''')

# ── stack ─────────────────────────────────────────────────────────────────────
conf   = str(work_dir/'Caddyfile')
dc_path = str(work_dir/'docker-compose.yml')

orig = os.getcwd(); os.chdir(work_dir)
try:
    dc = tenant_stack(BASE_DOMAIN, tenants,
                      db=str(work_dir/'routes.db'), conf=conf)
    dc.save(dc_path)
    print(dc)
    print('--- Caddyfile ---')
    print(Path(conf).read_text())

    dc.up(path=dc_path)
    print('Waiting for image build + tunnel...')
    results = {}
    for name in ('acme', 'globex', 'initech'):
        url = f'https://{name}.{BASE_DOMAIN}'
        for i in range(18):
            time.sleep(10)
            try: results[name] = urlread(url); break
            except Exception as e: print(f'  [{(i+1)*10}s] {name}: {e}')

    print('=== logs ===')
    print(dc.logs(path=dc_path))
    for name in ('acme', 'globex', 'initech'):
        assert name in results, f'{name} never responded'
        assert f'routed to {name}' in results[name], f'{name} wrong response'
        print(f'✓ {name}.{BASE_DOMAIN} → {name}')
finally:
    dc.down(path=dc_path, v=True, remove_orphans=True)
    os.chdir(orig)
    print('Cleaned up.')